# SOLiD-PINN v0.8.4 — Google Drive return (no GitHub token)

Run all, authorize Drive if prompted. By default recover the latest complete existing v0.8.4 result; do not recompute. No GitHub authentication or write. Public aggregate summary only is placed in a uniquely named handoff file in Drive. If ChatGPT's Drive connection cannot see it, download that small JSON only. To choose another existing run set RUN_ID. To deliberately start a fresh computation set RUN_NEW=True. This is a synthetic analytical Bayesian reference, not experimental validation or Bayesian PINN.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import os, sys, tempfile, subprocess, json, uuid, hashlib, importlib.util
drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive/SOLiD-PINN/colab_runs/v084")
ROOT.mkdir(parents=True,exist_ok=True)
RUN_ID="" # blank = latest complete valid run
RUN_NEW=False # change to True only to deliberately run a new benchmark
REV="1b4223f43aeab236a9868ed71c5bb46c21b7c028"
REPO=Path(tempfile.mkdtemp(prefix="solid-pinn-drive-"))/"repo"
subprocess.run(["git","clone","--no-checkout","https://github.com/theerayutbo/SOLiD-PINN.git",str(REPO)],check=True)
subprocess.run(["git","checkout","--detach",REV],cwd=REPO,check=True)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip()==REV
spec=importlib.util.spec_from_file_location("verified_pub",REPO/"scripts/publish_colab_uq.py")
pub=importlib.util.module_from_spec(spec)
spec.loader.exec_module(pub)
solver=(REPO/"scripts/bayes_interface.py").read_bytes()
tests=(REPO/"scripts/test_bayes_interface.py").read_bytes()
assert pub.git_blob(solver)==pub.SOURCE_BLOB and pub.git_blob(tests)==pub.TEST_BLOB
print("Verified source",REV)

## Recover or deliberately compute
Recovery validates the complete saved report and compares the existing summary byte-for-byte. An invalid report is not silently replaced or published. A new run is optional, off by default.

In [ ]:
def atomic_new(path, data):
    if path.exists():
        if path.read_bytes()!=data: raise RuntimeError("Existing file differs; refusing overwrite: "+path.name)
        return
    part=path.with_name(path.name+"."+uuid.uuid4().hex+".part")
    part.write_bytes(data)
    part.replace(path)
    assert path.read_bytes()==data

if RUN_NEW:
    if RUN_ID: raise ValueError("Clear RUN_ID before starting a new run")
    RUN=ROOT/(datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"-"+uuid.uuid4().hex[:8])
    RUN.mkdir()
    env=os.environ.copy()
    env.update(OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1",NUMEXPR_NUM_THREADS="1")
    subprocess.run([sys.executable,"-m","pip","install","numpy==2.3.5","scipy==1.17.0","pytest","requests"],check=True)
    subprocess.run([sys.executable,"-m","pytest","-q","scripts/test_bayes_interface.py","scripts/test_publish_colab_uq.py"],cwd=REPO,env=env,check=True)
    with tempfile.TemporaryDirectory() as tmp:
        out=Path(tmp)/"v084_full.json"
        subprocess.run([sys.executable,"scripts/bayes_interface.py","--out",str(out),"--n","151","--draws","16000","--repeats","30"],cwd=REPO,env=env,check=True)
        raw=out.read_bytes()
        bundle=pub.build_bundle(raw,REV,solver,tests)
        atomic_new(RUN/"v084_full.json",raw)
        atomic_new(RUN/"v084_summary.json",pub.canonical(bundle))
else:
    if RUN_ID and (Path(RUN_ID).name!=RUN_ID or RUN_ID in (".","..")):
        raise ValueError("RUN_ID must be one folder name")
    candidates=[ROOT/RUN_ID] if RUN_ID else sorted((p for p in ROOT.iterdir() if p.is_dir() and (p/"v084_full.json").is_file()),key=lambda p:p.name,reverse=True)
    if not candidates: raise RuntimeError("No saved full report found. Check Drive account or set RUN_NEW=True explicitly.")
    RUN=candidates[0]
    raw=(RUN/"v084_full.json").read_bytes()
    bundle=pub.build_bundle(raw,REV,solver,tests)
    atomic_new(RUN/"v084_summary.json",pub.canonical(bundle))
    print("Recovered existing run; no recomputation:",RUN.name)
print("Completed:",bundle["completed"])

## Save verified handoff
No GitHub writes, credentials, or private source data. The result is uniquely named by its SHA-256. ChatGPT can retrieve the file if the connected Drive account has access. The optional download cell is a fallback.

In [ ]:
HANDOFF=ROOT/"handoff"
HANDOFF.mkdir(exist_ok=True)
payload=pub.canonical(bundle)
sha=hashlib.sha256(payload).hexdigest()
DEST=HANDOFF/f"SOLiD-PINN_v084_colab_{sha[:20]}.json"
atomic_new(DEST,payload)
assert hashlib.sha256(DEST.read_bytes()).hexdigest()==sha
print("DRIVE VERIFIED")
print("File:",DEST.name)
print("Drive path:",DEST)
print("SHA256:",sha)
print("Completed:",bundle["completed"])
print("Coverage:",json.dumps(bundle["coverage"],indent=2))

Optional fallback: run the next cell only if the Drive connector cannot find the file. It downloads only the aggregate JSON, not the full report.

In [ ]:
# Uncomment only if a manual upload is necessary:
# from google.colab import files
# files.download(str(DEST))